# March 18 flat-dir trigger-mode benchmark

This notebook is separate from the synthetic periodic-branch benchmark. It scores the real March 18 flat-directory `.dat3` light curves, uses the full per-camera GP baseline (`baseline_func="gp"`), and leaves residual-space bad-camera filtering enabled so offset/scatter cameras are removed after the first GP fit and then refit.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "malca").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "malca").is_dir():
    raise RuntimeError("Could not find repo root containing ./malca")
os.chdir(repo_root)

from malca.evaluation.march18_flat_trigger_mode_benchmark import (
    March18FlatTriggerConfig,
    load_march18_flat_trigger_mode_benchmark,
    plot_score_space,
    plot_threshold_sweep,
    run_march18_flat_trigger_mode_benchmark,
)

pd.set_option("display.max_columns", 120)
repo_root

## Existing March 18 runs

`local_lc_all_run` already used the real March 18 flat light curves, but that run used `gp_masked` and skipped the raw camera-median audit. This notebook creates a separate real-light-curve run with full GP correction.

In [ ]:
def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

existing_runs = {
    "local_lc_all_run": repo_root / "output/runs/local_lc_all_run/run_params.json",
    "march18_bundle_all": repo_root / "output/runs/runs_march18_bundle_all/run_params.json",
}

for label, path in existing_runs.items():
    print(f"\n{label}: {path}")
    if not path.exists():
        print("  missing")
        continue
    params = read_json(path)
    for key in [
        "flat_lc_dir",
        "baseline_func",
        "trigger_mode",
        "skip_camera_median",
        "filter_bad_cameras",
        "stage",
        "workers",
    ]:
        if key in params:
            print(f"  {key}: {params[key]}")

## Run controls

By default this runs the full `.dat3` manifest. Set `MALCA_MARCH18_FLAT_SMOKE=1` for a 25-source smoke run, `MALCA_MARCH18_FLAT_MAX_SOURCES=N` for a sampled run, or `MALCA_MARCH18_FLAT_FORCE=1` to overwrite cached outputs for the same tag.

In [ ]:
FLAT_LC_DIR = repo_root / "output/runs/runs_march18_bundle_all/bundle_assets/lightcurves"
MANIFEST_PATH = repo_root / "output/runs/local_lc_all_run/manifests/lc_manifest_all.parquet"
INDEX_FILE = repo_root / "output/runs/local_lc_all_magbin_index.parquet"
OUTPUT_BASE_DIR = repo_root / "output/diagnostics/march18_flat_trigger_mode_benchmark"

RUN_TAG = os.environ.get("MALCA_MARCH18_FLAT_RUN_TAG", "march18_flat_full_gp_real")
SMOKE = os.environ.get("MALCA_MARCH18_FLAT_SMOKE", "0") == "1"
MAX_SOURCES_ENV = os.environ.get("MALCA_MARCH18_FLAT_MAX_SOURCES")
MAX_SOURCES = int(MAX_SOURCES_ENV) if MAX_SOURCES_ENV else (25 if SMOKE else None)
WORKERS = int(os.environ.get("MALCA_MARCH18_FLAT_WORKERS", "8"))
FORCE = os.environ.get("MALCA_MARCH18_FLAT_FORCE", "0") == "1"

print(f"flat_lc_dir: {FLAT_LC_DIR}")
print(f"manifest_path: {MANIFEST_PATH}")
print(f"dat3 files: {len(list(FLAT_LC_DIR.glob('*.dat3'))) if FLAT_LC_DIR.exists() else 0}")
print(f"run_tag: {RUN_TAG}")
print(f"max_sources: {MAX_SOURCES}")
print(f"workers: {WORKERS}")
print(f"force: {FORCE}")

In [ ]:
config = March18FlatTriggerConfig(
    output_base_dir=OUTPUT_BASE_DIR,
    run_tag=RUN_TAG,
    flat_lc_dir=FLAT_LC_DIR,
    manifest_path=MANIFEST_PATH,
    index_file=INDEX_FILE,
    extension="dat3",
    max_sources=MAX_SOURCES,
    workers=WORKERS,
    force=FORCE,
    baseline_func="gp",
    auto_filter_bad_cameras=True,
    event_kinds=("dip", "jump"),
)

config

## Score real light curves

The scorer computes the full GP baseline once per light curve, applies residual bad-camera filtering, refits when cameras are removed, and then sweeps posterior-probability and local-logBF thresholds from the same scored arrays.

In [ ]:
run = run_march18_flat_trigger_mode_benchmark(config)

print(f"run_dir: {run.run_dir}")
display(pd.DataFrame([
    {"table": "manifest", "rows": len(run.manifest), "columns": len(run.manifest.columns)},
    {"table": "score_results", "rows": len(run.score_results), "columns": len(run.score_results.columns)},
    {"table": "trigger_results", "rows": len(run.trigger_results), "columns": len(run.trigger_results.columns)},
]))
display(run.score_results["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows"))
display(run.pairwise_production)

In [ ]:
summary = run.summary_overall.sort_values(["event_kind", "trigger_family", "threshold"])
display(summary)

summary_path = run.run_dir / "summary_tables" / "all_thresholds_overall.csv"
pairwise_path = run.run_dir / "summary_tables" / "pairwise_production.csv"
print(f"summary: {summary_path}")
print(f"pairwise: {pairwise_path}")

## Threshold sweeps

In [ ]:
from malca.lightcurve_publication import finalize_publication_figure
for kind in config.event_kinds:
    axes = plot_threshold_sweep(run.summary_overall, kind=kind)
    fig = axes[0, 0].figure
    fig.suptitle(f"March 18 real LC threshold sweep: {kind}", y=1.02)
    finalize_publication_figure(fig)
    out = run.run_dir / f"threshold_sweep_{kind}.png"
    fig.savefig(out, dpi=180)
    print(out)


## Score space

In [ ]:
from malca.lightcurve_publication import finalize_publication_figure
fig, axes = plt.subplots(1, len(config.event_kinds), figsize=(7 * len(config.event_kinds), 5), squeeze=False)
for axis, kind in zip(axes[0], config.event_kinds):
    plot_score_space(run.score_results, kind=kind, ax=axis)
finalize_publication_figure(fig)
out = run.run_dir / "score_space.png"
fig.savefig(out, dpi=180)
print(out)


## Production-threshold disagreements

In [ ]:
def production_disagreements(kind: str, *, limit: int = 25) -> pd.DataFrame:
    prod = run.trigger_results[
        run.trigger_results["status"].eq("ok")
        & run.trigger_results["is_production"].fillna(False).astype(bool)
        & run.trigger_results["event_kind"].eq(kind)
    ].copy()
    pp = prod[prod["trigger_family"].eq("loo_posterior_prob")].set_index("source_id")
    bf = prod[prod["trigger_family"].eq("local_logbf")].set_index("source_id")
    cols = ["significant", "event_points", "trigger_max", "max_event_probability", "max_log_bf_local", "dat_path"]
    if pp.empty or bf.empty:
        return pd.DataFrame()
    wide = pp[cols].join(bf[cols], how="outer", lsuffix="_posterior", rsuffix="_logbf")
    posterior_sig = wide["significant_posterior"].fillna(False).astype(bool)
    logbf_sig = wide["significant_logbf"].fillna(False).astype(bool)
    wide["case"] = np.select(
        [posterior_sig & logbf_sig, posterior_sig & ~logbf_sig, ~posterior_sig & logbf_sig],
        ["both", "posterior_only", "logbf_only"],
        default="neither",
    )
    return (
        wide[wide["case"].ne("neither")]
        .sort_values(["case", "max_event_probability_posterior", "max_log_bf_local_posterior"], ascending=[True, False, False])
        .head(limit)
        .reset_index()
    )

for kind in config.event_kinds:
    print(kind)
    display(production_disagreements(kind))

## Full pipeline command

This notebook is the trigger-mode benchmark. If you want a separate full `malca pipeline` run on the same real March 18 flat directory, use `--baseline-func gp` and do not reuse `local_lc_all_config.json`, because that config has `skip_camera_median=true`.

In [ ]:
PIPELINE_OUTPUT_DIR = repo_root / "output/runs/march18_flat_full_gp_run"
pipeline_cmd = " ".join([
    "python -m malca pipeline",
    "--mag-bin all",
    f"--flat-lc-dir {FLAT_LC_DIR}",
    f"--index-file {INDEX_FILE}",
    f"--output-dir {PIPELINE_OUTPUT_DIR}",
    "--stage cluster",
    f"--workers {WORKERS}",
    "--baseline-func gp",
    "--trigger-mode posterior_prob",
    "-v",
])
print(pipeline_cmd)